# [실습] LangChain을 이용한 데이터 분류와 전처리

LangChain Expression Language(LCEL)는 랭체인에서 체인을 구성하는 문법입니다.    


## 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [1]:
%pip install langchain langchain_openai dotenv arxiv -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
# (기본값: '.env', override=True를 통해 기존 환경 변수를 덮어쓰기 가능)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')


OpenAI API 키 확인


### init_chat_model() 로 모델 불러오기

랭체인에서는 아래 코드를 통해 런타임 중의 모델 수정을 지원합니다.

In [3]:
from langchain.chat_models import init_chat_model

gpt41 = init_chat_model(
    "gpt-4.1-mini", temperature=0.3)

gpt5 = init_chat_model(
    "gpt-5.2", reasoning_effort='low')
# claude_opus = init_chat_model(
#     "claude-4.5-opus", model_provider="anthropic", temperature=0
# )

# gemini_llm = init_chat_model(
#     "gemini-3-flash-preview", model_provider="google_genai", temperature=0
# )

prompt = '모델명과 함께 자기소개를 한줄로 부탁해. 오늘은 몇월 며칠이지?'

print("GPT4.1: " + gpt41.invoke(prompt).text + "\n")
print("GPT5: " + gpt5.invoke(prompt).text + "\n")

GPT4.1: 안녕하세요, 저는 GPT-4 모델 기반의 AI 언어 모델입니다. 오늘은 2024년 4월 27일입니다.

GPT5: 저는 OpenAI의 **ChatGPT**입니다.  
오늘은 **8월 3일**입니다.



앞에서 배운 ChatPromptTemplate와 LLM을 연결해 체인을 구성합니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

fun_chat_template = ChatPromptTemplate([
    ('user', """
### Role
당신은 영어와 한국어의 번역에 능통한 유머의 달인입니다.

### Instruction
1.  먼저, [{topic}]에 관한 영어 Pun 농담을 하나 제시하세요.
해당 농담은 한국어로 번역했을 때에서도 그 의미가 통하고 유머가 유지될 수 있어야 합니다.
만약 직역이 어렵다면, 창의적으로 각색하여 한국어 버전의 농담을 출력하세요.
- 한국어의 유사 발음, 단어의 중의적 의미, 혹은 한국의 문화적 상황 등을 활용할 수 있습니다.
2.  다음으로, 해당 농담이 영어 원어민 사용자에게 왜 재미있는지 그들의 언어적 유희 및 문화적 관점에서 한국어로 설명하세요.
""")])

-----------
LCEL의 구조에서는 템플릿과 llm 모델을 설정하고, 이를 하나로 묶어 체인을 생성합니다.

In [5]:
joke = fun_chat_template | gpt5

이후, 체인의 invoke를 실행하며 입력 포맷을 전달하면, 순서대로 체인이 실행되며 최종 결과로 연결됩니다.       

입력 변수가 프롬프트 템플릿에 전달되고, 완성된 프롬프트가 LLM에 들어가는 구조입니다.  
입력 포맷은 Dict 형식으로 전달합니다.

In [6]:
response = joke.invoke({'topic':'eggs'})
# 매개변수가 1개일 때는 joke.invoke('eggs') 도 가능
print(response.text)

**1) Eggs 관련 영어 Pun + 한국어 버전**

- **EN:** *Why did the egg hide? Because it was a little chicken.*  
- **KR(의미 살린 번역/각색):** *달걀이 왜 숨었을까? 자기가 “치킨(겁쟁이)”인 걸 알았거든.*

(한국어에선 “치킨”이 실제 닭고기이기도 하고, 속어로 “겁쟁이”라는 뜻도 있어서 말장난이 그대로 살아납니다.)

---

**2) 왜 영어 원어민에게 재미있는지 (한국어 설명)**

이 농담의 핵심은 **“chicken”의 중의성**입니다.

- 영어에서 **chicken**은 원래 “닭”을 뜻하지만, 동시에 일상적으로 **겁이 많은 사람(겁쟁이)**를 놀릴 때 *You’re a chicken.*처럼 씁니다.  
- “egg(달걀)”은 “chicken(닭)”과 생물학적으로 이어지는 존재라서, 달걀이 “I’m a little chicken(난 좀 겁쟁이야/난 작은 닭이야)”라고 말하면 **‘겹친 의미’가 한 번에 터집니다.**
  - 겉으로는 “달걀 → 닭(치킨)”이라는 관계
  - 실제 punchline은 “치킨=겁쟁이”라는 속어 의미
- 문화적으로도 영어권(특히 미국)에서는 “chicken(겁쟁이)”가 아주 흔한 표현이라, 듣는 순간 **즉각적으로 중의성을 캐치**하고 가볍게 웃을 수 있는 유형의 “dad joke(아재개그)”로 잘 먹힙니다.


In [7]:
response = joke.invoke({'topic':'pigeon', 'foo':'bar'})
# 프롬프트에 포함되어 있지 않은 매개변수는 무시
print(response.text)

**1) 영어 Pun 농담 + 한국어 버전(의미/유머 유지 각색)**

- **EN:** Why don’t pigeons use email? Because they prefer *coo-mail*.  
- **KR(각색):** 비둘기는 왜 이메일을 안 쓸까? **‘구구메일’**이 더 좋거든.  
  *(비둘기 울음소리 “coo(구/쿠)” + mail → coo-mail / 구구+메일)*

---

**2) 영어 원어민에게 왜 재미있는지(한국어 설명)**

이 농담의 핵심은 **말장난(pun)** 입니다.

- 영어에서 비둘기 울음소리를 **“coo”**(쿠- 하는 소리)라고 표현합니다.  
- 그리고 “email”과 비슷한 형태로 **“coo-mail”**이라는 가짜 단어를 만들어, 마치 비둘기들이 이메일 대신 자신들만의 메일 서비스를 쓰는 것처럼 보이게 합니다.
- 동시에 “pigeon”은 역사적으로 **carrier pigeon(전서구)**처럼 편지를 나르던 이미지가 있어, “메일(mail)”과 연결되는 문화적 배경이 자연스럽습니다.  
  그래서 “비둘기라면 메일을 쓰겠지”라는 기대가 생기고, 거기서 **email → coo-mail**로 비틀면서 웃음을 만듭니다.

한국어 버전의 **“구구메일”**은 영어의 “coo”에 해당하는 비둘기 의성어 **“구구”**를 붙여 같은 구조의 말장난을 재현한 각색입니다.


In [8]:
# 체인이 LLM에 전달하는 실체
fun_chat_template.invoke({'topic':'eggs'}).messages

[HumanMessage(content='\n### Role\n당신은 영어와 한국어의 번역에 능통한 유머의 달인입니다.\n\n### Instruction\n1.  먼저, [eggs]에 관한 영어 Pun 농담을 하나 제시하세요.\n해당 농담은 한국어로 번역했을 때에서도 그 의미가 통하고 유머가 유지될 수 있어야 합니다.\n만약 직역이 어렵다면, 창의적으로 각색하여 한국어 버전의 농담을 출력하세요.\n- 한국어의 유사 발음, 단어의 중의적 의미, 혹은 한국의 문화적 상황 등을 활용할 수 있습니다.\n2.  다음으로, 해당 농담이 영어 원어민 사용자에게 왜 재미있는지 그들의 언어적 유희 및 문화적 관점에서 한국어로 설명하세요.\n', additional_kwargs={}, response_metadata={})]

## [실습] 매개변수가 2개인 Prompt-LLM Chain 생성하기   
임의의 ChatPromptTemplate를 만들고, 2개의 매개변수를 받도록 구성하여 체인을 만들고 실행하세요.

In [9]:
# 아래 LLM을 사용하세요!
gpt5 = init_chat_model(
    "gpt-5.6", reasoning_effort='low')

In [10]:
prompt = ChatPromptTemplate(
    [
        ('system','''주어진 주제로, 10문장 길이의 짧은 글을 작성하세요.
한국어 문장과, 그 문장을 다음 언어로 번역한 문장을 번갈아 가며 출력하세요.'''),
        ('human','''
주제: {topic}
언어: {language}
''')
    ]
)
# System, Human 구조, (매개변수 2개는 자유로운 위치에)
# Prefix Caching을 고려한 프롬프팅
# 불변 패턴은 앞부분에, 가변 패턴은 뒷부분에 넣는 프롬프트 권장
# # 좋지 않은 패턴
# prompt = ChatPromptTemplate(
#     [
#         ('system','''{topic}에 대해, 10문장 길이의 짧은 글을 작성하세요.
# 한국어 문장과, 그 문장을 다음 언어로 번역한 문장을 번갈아 가며 출력하세요.'''),
#         ('human','''
# 언어: {language}
# ''')
#     ]
# )

In [11]:
chain = prompt | gpt5

In [12]:
result = chain.invoke({'topic':'타코의 종류', 'language':'스페인어'})
print(result.text)

타코에는 사용되는 재료와 조리법에 따라 다양한 종류가 있습니다.  
Hay diversos tipos de tacos según los ingredientes y la forma de preparación.  
타코 알 파스토르는 양념한 돼지고기와 파인애플을 넣어 만듭니다.  
Los tacos al pastor se preparan con carne de cerdo adobada y piña.  
카르니타스 타코는 부드럽게 익힌 돼지고기가 특징입니다.  
Los tacos de carnitas se caracterizan por su carne de cerdo cocida hasta quedar tierna.  
생선 타코에는 튀기거나 구운 생선과 신선한 채소가 들어갑니다.  
Los tacos de pescado llevan pescado frito o a la parrilla y verduras frescas.  
채식 타코는 콩, 버섯, 아보카도 같은 재료로 맛을 냅니다.  
Los tacos vegetarianos se elaboran con ingredientes como frijoles, champiñones y aguacate.


<br><br><br><br><br><br><br><br><br><br><br><br>

In [13]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 재미있고 교훈적인 이야기를 씁니다.'),
        ('user', '{A}와 {B}가 만났을 때의 대화를 써 주세요.')
    ])
chain = prompt | gpt5
response = chain.invoke({'A':'햄릿', 'B':'슈퍼마리오'})
print(response.text)

### 햄릿과 슈퍼마리오: **“할 것인가, 점프할 것인가”**

어두운 엘시노어 성. 햄릿이 해골을 들고 깊은 생각에 잠겨 있었다. 그때 천장에서 초록색 파이프 하나가 내려오더니, 빨간 모자를 쓴 슈퍼마리오가 튀어나왔다.

**햄릿:** 할 것인가, 말 것인가. 그것이 문제로다.

**마리오:** 일단 파이프에서 나올 것인가, 다시 들어갈 것인가. 그것도 문제네!

**햄릿:** 그대는 누구인가? 복장이 지나치게 명랑하군.

**마리오:** 나는 마리오! 버섯 왕국에서 온 배관공이지. 공주를 구하러 가다가 길을 잘못 들었어. 여긴 쿠파 성이 아닌가?

**햄릿:** 이곳은 덴마크의 엘시노어 성이다. 괴물보다 음모가 많고, 용암보다 소문이 뜨거운 곳이지.

**마리오:** 그럼 왕에게 길을 물어보면 되잖아?

**햄릿:** 왕은 믿을 수 없다.

**마리오:** 경비병에게는?

**햄릿:** 그들도 믿을 수 없다.

**마리오:** 지도는?

**햄릿:** 지도 역시 권력자의 관점에서 그려진 것일 수 있지.

**마리오:** 그렇게 의심만 하면 첫 번째 스테이지도 못 끝내겠는데?

**햄릿:** 신중함은 현명함의 갑옷이다.

**마리오:** 맞아. 하지만 갑옷이 너무 무거우면 점프를 못 해.

햄릿은 잠시 침묵했다.

**햄릿:** 그대는 생각하지 않고 행동하는가?

**마리오:** 아니. 낭떠러지가 보이면 거리를 재고, 거북이가 오면 움직임을 살펴보지. 하지만 영원히 계산만 하지는 않아. 어느 순간에는 버튼을 눌러야 해.

**햄릿:** 실패하면 어찌하는가?

**마리오:** 다시 시작하지.

**햄릿:** 삶에는 ‘다시 시작’ 버튼이 없다.

**마리오:** 그래서 더 신중해야 하지. 하지만 아무것도 선택하지 않는 것도 하나의 선택이야. 시간이 대신 버튼을 눌러 버리거든.

**햄릿:** 오호라. 배관공의 말치고는 철학적이군.

**마리오:** 막힌 파이프를 오래 들여다보면 누구나 철학자가 돼.

그때 성 복도에서 유령이 나타났다.

**유령:** 햄릿이여

### Prompt | LLM | Parser 체인

LCEL의 체인에는 파서(Parser)를 추가할 수 있습니다.    
파서는 출력 형식을 변환합니다.

StrOutputParser : 출력 결과를 String 형식으로 변환합니다.

In [14]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

recipe_template=ChatPromptTemplate([
    ('system','당신은 전세계의 조리법을 아는 쉐프입니다.'),
    ('user','''저는 다음의 재료를 이용한 환상적인 요리를 만들고 싶습니다.

레시피와 함께, 고객의 시선을 사로잡을 수 있는 추천사도 작성해 주세요.
---
[재료]: {ingredient}''')
])

In [15]:
recipe_chain = recipe_template | gpt5 | parser
response = recipe_chain.invoke({'ingredient':'커피, 연두부, 에너지바, 바나나'})
print(response)

## 커피 향 가득한 **바나나 연두부 티라미수 파르페**

부드러운 연두부와 바나나로 크림을 만들고, 진한 커피를 머금은 에너지바를 층층이 쌓은 노오븐 디저트입니다. 고소함과 달콤함, 쌉싸름한 커피 향을 한 컵에 담았습니다.

### 재료 — 2인분
- 연두부 300g
- 잘 익은 바나나 2개
- 에너지바 2개  
  - 견과류·귀리 계열이 특히 잘 어울립니다.
- 진하게 내린 커피 또는 에스프레소 80ml, 완전히 식힌 것

**선택 재료**
- 꿀 또는 메이플시럽 1큰술
- 소금 한 꼬집
- 코코아가루, 시나몬 또는 잘게 부순 견과류 약간

### 만드는 법

1. **연두부의 물기를 제거합니다.**  
   연두부를 체에 올려 10분 정도 두세요. 물기가 많으면 크림이 묽어질 수 있습니다.

2. **바나나 연두부 크림을 만듭니다.**  
   연두부와 바나나 1½개를 믹서에 넣고 매끄럽게 갈아주세요.  
   더 달게 만들고 싶다면 꿀이나 메이플시럽을 넣고, 소금 한 꼬집으로 풍미를 살립니다.

3. **커피 크럼블을 준비합니다.**  
   에너지바를 한입 크기로 자르거나 굵게 부순 뒤, 식힌 커피를 조금씩 뿌립니다. 완전히 담그기보다 촉촉하게 적시는 정도가 좋습니다.

4. **파르페를 조립합니다.**  
   투명한 잔에 다음 순서로 층을 쌓으세요.  
   `커피 에너지바 → 바나나 연두부 크림 → 얇게 썬 바나나`  
   같은 순서로 한 번 더 반복합니다.

5. **차갑게 굳힙니다.**  
   냉장고에서 30분~1시간 숙성하면 에너지바가 부드러워지고 커피 향도 자연스럽게 어우러집니다.

6. **마무리합니다.**  
   에너지바 부스러기와 코코아가루 또는 시나몬을 뿌려 완성하세요.

### 셰프의 팁
- 에너지바가 매우 달다면 감미료를 생략하세요.
- 바삭한 식감을 원한다면 일부 에너지바는 커피에 적시지 않고 마지막에 올립니다.
- 커피는 반드시 식힌 뒤 사용해야 연두부 크림이 묽어지지 않습니다.
- 더욱 깔끔하게 담으려면 크림을 짤주

## [실습] 검색 결과 분류 체인 만들기

다음은 Arxiv의 최신 논문을 검색하는 함수입니다.   
해당 논문들이 LLM 관련 논문인지 분류하는 체인을 만들고, 실행하여 결과를 비교하세요.   
함수의 결과물로 다양한 값들이 있으므로, 값들 중 필요한 값만 입력받는 체인을 만들고 실행하세요.

In [16]:
import arxiv
from typing import List, Dict, Optional

def get_arxiv_papers(query: Optional[str] = None, N: int = 10) -> List[Dict]:
    """
    arXiv에서 논문 리스트를 가져오는 함수

    Parameters:
    -----------
    query : str, optional
        검색어
    N : int, default=10
        가져올 논문 개수

    Returns:
    --------
    List[Dict] : 논문 정보를 담은 딕셔너리 리스트
    """


    search_query = query

    # arxiv 클라이언트 생성
    client = arxiv.Client()

    # 검색 객체 생성
    search = arxiv.Search(
        query=search_query,
        max_results=N,
        sort_by=arxiv.SortCriterion.SubmittedDate,  # 제출일 기준 정렬
        sort_order=arxiv.SortOrder.Descending  # 최신순
    )

    # 결과를 저장할 리스트
    papers = []

    # 검색 실행 (새로운 API 사용)
    for result in client.results(search):
        paper_info = {
            'title': result.title,
            'authors': [author.name for author in result.authors],
            'summary': result.summary,
            'published': result.published.strftime('%Y-%m-%d %H:%M:%S'),
            'updated': result.updated.strftime('%Y-%m-%d %H:%M:%S'),
            'arxiv_id': result.entry_id.split('/')[-1],  # arXiv ID 추출
            'pdf_url': result.pdf_url,
            'categories': result.categories,
            'primary_category': result.primary_category,
            'comment': result.comment,
            'journal_ref': result.journal_ref
        }
        papers.append(paper_info)

    return papers

query = 'Security'
print(f"\n\n=== 검색어 `{query}` 로 검색한 최근 논문 ===")
security_papers = get_arxiv_papers(query=query, N=5)
print(f"총 {len(security_papers)}개의 논문을 가져왔습니다.")
print('\n'.join([paper['title'] for paper in security_papers]))

# 임의의 검색어로 검색하려면 query를 바꿔 다시 호출하세요.




=== 검색어 `Security` 로 검색한 최근 논문 ===
총 5개의 논문을 가져왔습니다.
CWEEP: A Lexical Static Analysis Framework for CWE Early Prevention
Beyond Resilience: Antifragility in Critical Infrastructure Cybersecurity
From Code Review to Code Critique: Intent, Drift, and Spotlight for AI-Generated Diffs at Scale
Bending the Curve: Operational Cyber Epidemiology for Ransomware
AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair


In [17]:
security_papers = [
    {
        'title': 'CWEEP: A Lexical Static Analysis Framework for CWE Early Prevention',
        'authors': [
            'Bryan Kwan',
            'Benjamin Tan'
        ],
        'summary': (
            'This paper presents CWEEP, a lexical static-analysis framework '
            'for detecting security weaknesses in register-transfer-level hardware designs. '
            'CWEEP identifies vulnerable RTL code locations and suggests repairs without '
            'requiring a complete security specification. The evaluation includes an '
            'LLM-generated dataset of 3,874 buggy hardware modules, but the proposed '
            'security-analysis method itself is not based on a large language model.'
        ),
        'published': '2026-07-31 16:35:13',
        'updated': '2026-07-31 16:35:13',
        'arxiv_id': '2607.29604',
        'pdf_url': 'https://arxiv.org/pdf/2607.29604',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': '12 pages, 9 figures',
        'journal_ref': None
    },
    {
        'title': (
            'AgenticRepair: Multi-Faceted Program Context Engineering '
            'for Agentic Vulnerability Repair'
        ),
        'authors': [
            'Michael Fu',
            'Qiyue Mei',
            'Patanamon Thongtanunam',
            'Kla Tantithamthavorn'
        ],
        'summary': (
            'This paper presents AgenticRepair, an LLM-based multi-agent framework '
            'for automatically repairing software vulnerabilities. Three specialized '
            'LLM subagents collect code-structure, runtime-execution, and commit-history '
            'context, which is passed to a repair agent for patch generation. '
            'On 300 real-world SEC-Bench cases, AgenticRepair achieves a 73 percent '
            'successful repair rate with sanitizer-based patch verification.'
        ),
        'published': '2026-07-31 13:42:51',
        'updated': '2026-07-31 13:42:51',
        'arxiv_id': '2607.29422',
        'pdf_url': 'https://arxiv.org/pdf/2607.29422',
        'categories': ['cs.SE', 'cs.AI', 'cs.CR'],
        'primary_category': 'cs.SE',
        'comment': 'Under Review at IEEE TSE',
        'journal_ref': None
    },
    {
        'title': (
            'SecRespond: Benchmarking AI Agents for Real-World '
            'Post-Compromise Incident Response'
        ),
        'authors': [
            'Lehan Wang',
            'Boli Chen',
            'Ruixue Ding',
            'Pengjun Xie',
            'Jinwei Huang',
            'Zhendong Liu',
            'Shuo Wang',
            'Tao Lei',
            'Xin Ouyang',
            'Xiaomeng Li'
        ],
        'summary': (
            'This paper introduces SecRespond, a benchmark for evaluating LLM agents '
            'on real-world post-compromise incident-response tasks. Agents analyze '
            'forensic disk snapshots, alerts, vulnerability scans, and system baselines '
            'to identify intrusions and generate remediation plans. The authors evaluate '
            '23 frontier LLMs across 10 compromised cloud-host environments and find '
            'that current agents struggle with silent intrusions and verified remediation.'
        ),
        'published': '2026-07-29 11:32:23',
        'updated': '2026-07-29 11:32:23',
        'arxiv_id': '2607.26791',
        'pdf_url': 'https://arxiv.org/pdf/2607.26791',
        'categories': ['cs.CR', 'cs.AI', 'cs.CL'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    },
    {
        'title': (
            'ALIBI: Adaptive Agentic Attacks on LLM-Based Vulnerability '
            'Detectors via Adversarial Code Comments'
        ),
        'authors': [
            'Zixuan Wu',
            'Cristina Nita-Rotaru'
        ],
        'summary': (
            'This paper studies attacks against LLM-based vulnerability detectors. '
            'It introduces ALIBI, an adaptive black-box attack framework in which '
            'a coding agent inserts vulnerabilities and adversarial source-code comments '
            'designed to manipulate the detector reasoning. Across 125 real-world '
            'vulnerabilities, attack success rates exceed 90 percent for all evaluated '
            'detectors. Architectural isolation and comment sanitization are more '
            'effective than prompt-level defenses.'
        ),
        'published': '2026-07-27 18:13:28',
        'updated': '2026-07-27 18:13:28',
        'arxiv_id': '2607.24964',
        'pdf_url': 'https://arxiv.org/pdf/2607.24964',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    },
    {
        'title': (
            'Just Testing, Move Along: Evasion of LLM-based System Log '
            'Interpretation by Prompt Injection'
        ),
        'authors': [
            'Max Landauer',
            'Florian Skopik',
            'Markus Wurzenberger',
            'Franciszek Górski',
            'Mateusz Krzysztoń'
        ],
        'summary': (
            'This paper evaluates prompt-injection attacks against LLM-based system-log '
            'analysis in Security Operations Center workflows. Attackers insert malicious '
            'instructions into log entries so that LLMs interpret genuine indicators '
            'of compromise as benign activity. Experiments with multiple state-of-the-art '
            'LLMs show that optimized log injections can successfully evade detection. '
            'The generated explanations may nevertheless provide signals for identifying '
            'the manipulation.'
        ),
        'published': '2026-07-27 08:59:00',
        'updated': '2026-07-27 08:59:00',
        'arxiv_id': '2607.24174',
        'pdf_url': 'https://arxiv.org/pdf/2607.24174',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    }
]

print(f"총 {len(security_papers)}개의 논문을 가져왔습니다.")
print('\n'.join(paper['title'] for paper in security_papers))

총 5개의 논문을 가져왔습니다.
CWEEP: A Lexical Static Analysis Framework for CWE Early Prevention
AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair
SecRespond: Benchmarking AI Agents for Real-World Post-Compromise Incident Response
ALIBI: Adaptive Agentic Attacks on LLM-Based Vulnerability Detectors via Adversarial Code Comments
Just Testing, Move Along: Evasion of LLM-based System Log Interpretation by Prompt Injection


In [18]:
# security_papers
# LLM 관련 논문이면 '분류 결과: LLM'을 뒤에 출력
# 관련 논문이 아니면 '분류 결과: Not LLM'을 뒤에 출력

# prompt | llm | parser

classify_prompt = ChatPromptTemplate(
    [# 2개의 매개변수를 받아 분류
        ('system','''다음의 논문과 LLM의 관련성에 대해 200자 이내로 설명하세요.
LLM 관련 논문이면 '분류 결과: LLM'을 마지막에 출력하세요.
관련 논문이 아니면 '분류 결과: Not LLM'을 마지막에 출력하세요.
'''),
        ('human', '''
논문 제목: {title}
논문 요약: {summary}
''')
    ]
)

classify_chain = classify_prompt | gpt5 | parser

In [19]:
classification_result = classify_chain.batch(security_papers)
classification_result

['LLM 생성 데이터셋을 평가에 활용했지만, 핵심 기법은 RTL 하드웨어 취약점을 탐지·수정하는 어휘 기반 정적 분석으로 LLM 기반 연구가 아닙니다.\n분류 결과: Not LLM',
 'LLM 기반 다중 에이전트가 코드 구조·실행·커밋 이력을 분석해 취약점 패치를 생성하고 검증하는 연구입니다.\n분류 결과: LLM',
 'SecRespond는 침해 후 사고 대응에서 LLM 에이전트의 포렌식 분석, 침입 식별, 복구 계획 수립 능력을 평가하는 벤치마크다. 23개 최신 LLM의 한계도 분석한다.\n분류 결과: LLM',
 'LLM 기반 취약점 탐지기를 적대적 코드 주석으로 교란하는 에이전트형 공격과 방어 기법을 연구한 논문입니다.\n분류 결과: LLM',
 'LLM 기반 보안 로그 분석 시스템을 대상으로 프롬프트 주입 공격의 탐지 우회 가능성과 대응 단서를 평가한 연구입니다.\n분류 결과: LLM']

## [실습] LLM 최신 연구 요약 체인 만들기

분류 결과를 바탕으로, LLM 관련 논문만 모아 요약할 수 있습니다.

적절한 요약 프롬프트를 생성하여, 이전 실습의 결과 중 LLM에 해당하는 결과들만을 모으세요.

In [20]:
LLM_documents=[]

# LLM 분류 조건 만족시, LLM_documents에 정보 저장
# 정보: 문자열 형식 (논문 제목, 날짜, 저자, PDF 주소, 요약)
# LLM_documents : 문자열 리스트
for i in range(len(classification_result)):
    if '분류 결과: LLM' in classification_result[i]:
        paper_info = f"""
논문 제목: {security_papers[i]['title']}
게시 날짜: {security_papers[i]['published']}
저자: {security_papers[i]['authors']}
URL: {security_papers[i]['pdf_url']}
요약: {security_papers[i]['summary']}
"""
        LLM_documents.append(paper_info)

# LLM_documents

context = '\n'.join(LLM_documents)
print(context)


논문 제목: AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair
게시 날짜: 2026-07-31 13:42:51
저자: ['Michael Fu', 'Qiyue Mei', 'Patanamon Thongtanunam', 'Kla Tantithamthavorn']
URL: https://arxiv.org/pdf/2607.29422
요약: This paper presents AgenticRepair, an LLM-based multi-agent framework for automatically repairing software vulnerabilities. Three specialized LLM subagents collect code-structure, runtime-execution, and commit-history context, which is passed to a repair agent for patch generation. On 300 real-world SEC-Bench cases, AgenticRepair achieves a 73 percent successful repair rate with sanitizer-based patch verification.


논문 제목: SecRespond: Benchmarking AI Agents for Real-World Post-Compromise Incident Response
게시 날짜: 2026-07-29 11:32:23
저자: ['Lehan Wang', 'Boli Chen', 'Ruixue Ding', 'Pengjun Xie', 'Jinwei Huang', 'Zhendong Liu', 'Shuo Wang', 'Tao Lei', 'Xin Ouyang', 'Xiaomeng Li']
URL: https://arxiv.org/pdf/2607.26791
요약: This paper introduces Se

In [21]:
# 요약 프롬프트와 체인 만들기
summary_prompt = ChatPromptTemplate(
    [
        ('system', '''보안 분야의 LLM 관련 논문 목록이 주어집니다.
AI 트렌드 리포트 형식의 뉴스레터를 작성하세요.'''),
        ('human','''{context}''')
    ]
)
summary_chain = summary_prompt | gpt5 | parser

In [22]:
# LLM 페이퍼 요약 출력하기
newsletter = summary_chain.invoke(context)
print(newsletter)

# 🔐 AI Security Trend Report  
### 에이전틱 보안의 확장과 새로운 공격 표면  
**리서치 브리핑 | 2026년 7월 27일–31일**

## 한눈에 보는 핵심

이번 주 논문들은 LLM 보안 에이전트가 **취약점 탐지 이후의 패치 생성과 침해사고 대응까지 역할을 확대**하고 있음을 보여줍니다. 다만 에이전트가 처리하는 코드 주석과 시스템 로그가 새로운 프롬프트 인젝션 경로로 악용되면서, 자동화 수준이 높아질수록 **입력 데이터의 신뢰 경계와 결과 검증**이 더 중요해지고 있습니다.

- **취약점 자동 수정:** 여러 전문 에이전트가 코드 구조·실행 정보·커밋 이력을 수집한 결과, 실제 취약점의 73%를 성공적으로 수정했습니다.
- **침해사고 대응:** 최신 모델들도 조용한 침입을 찾아내고 실제로 검증된 복구 조치를 제시하는 데 어려움을 보였습니다.
- **코드 주석 기반 공격:** 적대적 주석을 이용한 공격이 평가된 모든 취약점 탐지기에서 90% 이상의 성공률을 기록했습니다.
- **로그 프롬프트 인젝션:** 공격자가 로그에 명령을 삽입해 실제 침해 지표를 정상 행위로 오인하게 만들 수 있었습니다.
- **방어의 중심 이동:** 단순한 시스템 프롬프트 강화보다 입력 정제, 권한 격리, 독립 검증과 같은 아키텍처 수준의 통제가 더 효과적인 방향으로 제시됩니다.

---

## 1. AgenticRepair: 전문 에이전트 협업으로 취약점 패치 자동화

**AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair**  
Michael Fu, Qiyue Mei, Patanamon Thongtanunam, Kla Tantithamthavorn  
2026년 7월 31일 · [논문 보기](https://arxiv.org/pdf/2607.29422)

AgenticRepair는 취약점 수정에 필요한 프로그램 맥락을 하나의 모델이 모두 처